# Crack Spread TFT Optuna - PTP Position Sizing

Optimizes `CrackSpreadTFT` on the 15-minute crack-spread master clock and adds a
4-class `PredictionToPosition` head so validation is selected on trading metrics,
not just classification loss.


In [ ]:
sys.path.append('/home/nicho/CTAFlow')


from notebooks.crack_spread_tft_ptp_support import (
    CrackSpreadPTP,
    PTPLoss,
    SharpeScheduler,
    build_crack_ptp_param_groups,
    build_windowed_crack_samples,
    compute_trading_metrics,
    fit_final_crack_ptp,
    hybrid_selection_score,
    infer_spread_feature_cols,
    make_crack_loaders,
    prepare_crack_data,
    split_crack_samples,
)
from CTAFlow.models.deep_learning.multi_branch.tft.crack_spread_tft import CrackSpreadTFT
from CTAFlow.models.deep_learning.multi_branch.tft.c_mmtft import returns_to_classes

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = device.type == 'cuda'
print(f'Using device: {device}')

def set_seed(seed: int = 42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

DATA_ROOT = Path(r'/home/nicho/model_data')
TICKERS = ('CL', 'HO', 'RB')
TICKERS = ('CL', 'HO', 'RB')
BAR_MINUTES = 15
FORECAST_HORIZON_MINUTES = 60
DECODER_STEPS = FORECAST_HORIZON_MINUTES // BAR_MINUTES
VAL_START = pd.Timestamp('2024-01-01')
TARGET_MODE = 'logret'
TARGET_COL = f'y_fwd_{DECODER_STEPS}'
SESSION_ONLY = True
SAMPLE_SESSION = None
SAMPLE_STRIDE = 1
MAX_TRAIN_SAMPLES = 30000
MAX_VAL_SAMPLES = 8000
N_TRIALS = 30
NUM_EPOCHS = 16
WARMUP_EPOCHS = 3
RESULTS_PATH = Path('artifacts') / 'crack_spread_tft_ptp'
RESULTS_PATH.mkdir(parents=True, exist_ok=True)
STUDY_NAME = f'crack_spread_tft_ptp_{BAR_MINUTES}m_{FORECAST_HORIZON_MINUTES}m'


In [ ]:
prep, df_out, train_mask, target_cols, orderflow_frames, orderflow_cols = prepare_crack_data(
    data_root=DATA_ROOT,
    tickers=TICKERS,
    bar_minutes=BAR_MINUTES,
    target_mode=TARGET_MODE,
    steps_60m=DECODER_STEPS,
)
known_temporal_cols = prep.get_known_temporal_cols(tickers=TICKERS)
spread_feature_cols = infer_spread_feature_cols(df_out, TARGET_COL, known_temporal_cols)

print('Data summary:')
print(f'  rows={len(df_out):,}')
print(f'  spread_features={len(spread_feature_cols)}')
print(f'  known_temporal_features={len(known_temporal_cols)}')
print(f'  orderflow_features={len(orderflow_cols)}')
print(f'  target_col={TARGET_COL}')

target_series = df_out[TARGET_COL].dropna()
target_labels = returns_to_classes(torch.tensor(target_series.values, dtype=torch.float32), outer=1.0).numpy()
print('Target class counts (outer=1.0):', np.bincount(target_labels, minlength=4))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(target_series.values, bins=120, alpha=0.8)
axes[0].set_title(f'{TARGET_COL} distribution')
axes[1].bar(np.arange(4), np.bincount(target_labels, minlength=4))
axes[1].set_title('4-class target counts')
axes[1].set_xticks(np.arange(4))
plt.tight_layout()

WINDOW_CACHE = {}

def get_train_val_samples(encoder_steps: int):
    encoder_steps = int(encoder_steps)
    if encoder_steps not in WINDOW_CACHE:
        samples = build_windowed_crack_samples(
            df_out=df_out,
            orderflow_frames=orderflow_frames,
            target_col=TARGET_COL,
            known_temporal_cols=known_temporal_cols,
            tickers=TICKERS,
            encoder_steps=encoder_steps,
            decoder_steps=DECODER_STEPS,
            session_only=SESSION_ONLY,
            sample_session=SAMPLE_SESSION,
            stride=SAMPLE_STRIDE,
            require_all_assets=True,
        )
        train_samples, val_samples = split_crack_samples(
            samples,
            val_start=VAL_START,
            max_train_samples=MAX_TRAIN_SAMPLES,
            max_val_samples=MAX_VAL_SAMPLES,
        )
        WINDOW_CACHE[encoder_steps] = (train_samples, val_samples)
    return WINDOW_CACHE[encoder_steps]

train_preview, val_preview = get_train_val_samples(48)
print(f'Cached preview samples: train={len(train_preview):,}, val={len(val_preview):,}')


In [ ]:
def objective(trial: optuna.Trial) -> float:
    encoder_steps = trial.suggest_categorical('encoder_steps', [32, 48, 64])
    d_model = trial.suggest_categorical('d_model', [64, 128])
    n_heads = trial.suggest_categorical('n_heads', [2, 4])
    n_lstm_layers = trial.suggest_int('n_lstm_layers', 1, 2)
    dropout = trial.suggest_float('dropout', 0.05, 0.30)
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-3, log=True)
    weight_decay = trial.suggest_float('weight_decay', 1e-4, 5e-3, log=True)
    batch_size = trial.suggest_categorical('batch_size', [32, 48, 64])
    max_norm = trial.suggest_float('max_norm', 0.5, 1.0)
    aux_weight = trial.suggest_float('aux_weight', 0.001, 0.10, log=True)

    vsn_temperature = trial.suggest_float('vsn_temperature', 0.75, 2.5)
    vsn_min_weight = trial.suggest_float('vsn_min_weight', 0.00, 0.08)
    vsn_entropy_weight = trial.suggest_float('vsn_entropy_weight', 0.005, 0.10, log=True)
    asset_temperature = trial.suggest_float('asset_temperature', 0.75, 2.5)
    asset_min_weight = trial.suggest_float('asset_min_weight', 0.00, 0.10)
    asset_entropy_weight = trial.suggest_float('asset_entropy_weight', 0.005, 0.10, log=True)

    ptp_temperature = trial.suggest_float('ptp_temperature', 1.0, 3.0)
    ptp_lr_scale = trial.suggest_float('ptp_lr_scale', 0.5, 1.5)
    ce_weight = trial.suggest_float('ce_weight', 0.3, 2.0, log=True)
    pnl_weight = trial.suggest_float('pnl_weight', 0.3, 2.0, log=True)
    profit_scale = trial.suggest_float('profit_scale', 25.0, 200.0)
    outer_threshold = trial.suggest_float('outer_threshold', 0.75, 1.5)

    tc_cost = trial.suggest_float('tc_cost', 5e-5, 5e-4, log=True)
    init_direction_weight = trial.suggest_float('init_direction_weight', 0.5, 1.5)
    final_direction_weight = trial.suggest_float('final_direction_weight', 0.05, 0.3)
    init_reg_weight = trial.suggest_float('init_reg_weight', 0.1, 0.5)
    target_exposure = trial.suggest_float('target_exposure', 0.2, 0.6)
    downside_vol_weight = trial.suggest_float('downside_vol_weight', 0.02, 0.75, log=True)
    holding_weight = trial.suggest_float('holding_weight', 0.0, 0.5)
    exposure_asymmetry = trial.suggest_float('exposure_asymmetry', 1.0, 6.0)

    try:
        train_samples, val_samples = get_train_val_samples(encoder_steps)
        train_loader, val_loader = make_crack_loaders(
            train_samples=train_samples,
            val_samples=val_samples,
            batch_size=batch_size,
            tickers=TICKERS,
            orderflow_lookback=encoder_steps,
            num_workers=0,
            return_metadata=False,
        )
    except Exception as exc:
        print(f'Dataloader failed: {exc}')
        return -1e9

    base_model = CrackSpreadTFT(
        n_past_features=len(spread_feature_cols),
        n_known_features=len(known_temporal_cols),
        n_orderflow_feats=len(orderflow_cols),
        n_assets=len(TICKERS),
        encoder_steps=encoder_steps,
        decoder_steps=DECODER_STEPS,
        d_model=d_model,
        n_heads=n_heads,
        n_lstm_layers=n_lstm_layers,
        dropout=dropout,
        num_classes=4,
        vsn_temperature=vsn_temperature,
        vsn_min_weight=vsn_min_weight,
        vsn_entropy_weight=vsn_entropy_weight,
        asset_temperature=asset_temperature,
        asset_min_weight=asset_min_weight,
        asset_entropy_weight=asset_entropy_weight,
    )
    model = CrackSpreadPTP(
        base_model=base_model,
        ptp_temperature=ptp_temperature,
        dropout=dropout,
    ).to(device)

    loss_fn = PTPLoss(
        ce_weight=ce_weight,
        pnl_weight=pnl_weight,
        profit_scale=profit_scale,
        outer_threshold=outer_threshold,
        tc_cost=tc_cost,
        direction_weight=init_direction_weight,
        reg_weight=init_reg_weight,
        target_exposure=target_exposure,
        use_sortino=True,
        downside_vol_weight=downside_vol_weight,
        tc_in_sharpe=True,
        holding_weight=holding_weight,
        exposure_asymmetry=exposure_asymmetry,
    ).to(device)

    optimizer = optim.AdamW(
        build_crack_ptp_param_groups(
            model,
            base_lr=learning_rate,
            weight_decay=weight_decay,
            ptp_lr_scale=ptp_lr_scale,
        )
    )
    lr_scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=NUM_EPOCHS,
        eta_min=learning_rate * 0.05,
    )
    sharpe_sched = SharpeScheduler(
        warmup_epochs=WARMUP_EPOCHS,
        total_epochs=NUM_EPOCHS,
        initial_direction_weight=init_direction_weight,
        final_direction_weight=final_direction_weight,
        initial_reg_weight=init_reg_weight,
        final_reg_weight=0.05,
        initial_target_exposure=0.2,
        final_target_exposure=target_exposure,
        initial_holding_weight=0.0,
        final_holding_weight=holding_weight,
    )
    scaler = torch.amp.GradScaler() if USE_AMP else None

    best_score = -1e9
    patience = 0
    prev_val_loss = None

    from notebooks.crack_spread_tft_ptp_support import train_epoch_crack_ptp, evaluate_crack_ptp

    for epoch in range(NUM_EPOCHS):
        sharpe_sched.step(epoch, loss_fn.trading_loss)
        train_loss, _ = train_epoch_crack_ptp(
            model=model,
            loader=train_loader,
            loss_fn=loss_fn,
            optimizer=optimizer,
            device=device,
            max_norm=max_norm,
            aux_weight=aux_weight,
            scaler=scaler,
        )
        val_metrics = evaluate_crack_ptp(
            model=model,
            loader=val_loader,
            loss_fn=loss_fn,
            device=device,
            aux_weight=aux_weight,
        )
        lr_scheduler.step()
        val_score = hybrid_selection_score(val_metrics, len(val_loader.dataset))

        if math.isnan(val_metrics['loss']) or math.isinf(val_metrics['loss']) or val_metrics['loss'] > 100.0:
            return best_score if best_score > -1e9 else -1e9
        if prev_val_loss is not None and epoch >= 3 and val_metrics['loss'] > abs(prev_val_loss) * 5.0:
            return best_score if best_score > -1e9 else -1e9
        prev_val_loss = val_metrics['loss']

        print(
            f"E{epoch + 1:02d} | Score {val_score:.4f} | Sharpe {val_metrics['sharpe']:.3f} | "
            f"Sortino {val_metrics['sortino']:.3f} | PF {val_metrics['profit_factor']:.3f} | "
            f"ClsAcc {val_metrics.get('cls_accuracy', 0.0):.2f}"
        )

        if epoch >= WARMUP_EPOCHS and val_score > best_score:
            best_score = val_score
            patience = 0
            trial.set_user_attr('final_selection_score', float(val_score))
            trial.set_user_attr('final_sharpe', float(val_metrics['sharpe']))
            trial.set_user_attr('final_sortino', float(val_metrics['sortino']))
            trial.set_user_attr('final_pf', float(val_metrics['profit_factor']))
            trial.set_user_attr('final_cls_acc', float(val_metrics.get('cls_accuracy', 0.0)))
            trial.set_user_attr('encoder_steps', int(encoder_steps))
        elif epoch >= WARMUP_EPOCHS:
            patience += 1

        trial.report(val_score, epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()
        if patience >= 6:
            break

    del model, base_model, optimizer, loss_fn, train_loader, val_loader
    if device.type == 'cuda':
        torch.cuda.empty_cache()
    gc.collect()
    return best_score


In [ ]:
study = optuna.create_study(
    study_name=STUDY_NAME,
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=4),
)
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True, gc_after_trial=True)

best_trial = study.best_trial
best_params = dict(best_trial.params)
best_params['best_value'] = best_trial.value
print('Best trial:', best_trial.number)
print(json.dumps(best_params, indent=2, default=str))

joblib.dump(study, RESULTS_PATH / f'{STUDY_NAME}_study.pkl')
df_trials = study.trials_dataframe()
df_trials.to_csv(RESULTS_PATH / f'{STUDY_NAME}_trials.csv', index=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
valid_trials = df_trials[df_trials['state'] == 'COMPLETE']
axes[0].plot(valid_trials.index, valid_trials['value'], 'o-', alpha=0.7)
axes[0].set_title('Optimization history')
if 'params_encoder_steps' in valid_trials.columns:
    axes[1].scatter(valid_trials['params_encoder_steps'], valid_trials['value'], c=valid_trials.index, cmap='viridis', s=70)
    axes[1].set_xlabel('encoder_steps')
axes[1].set_title('Score vs encoder_steps')
plt.tight_layout()


In [ ]:
best = dict(study.best_trial.params)
encoder_steps = int(best['encoder_steps'])
train_samples, val_samples = get_train_val_samples(encoder_steps)
train_loader, val_loader = make_crack_loaders(
    train_samples=train_samples,
    val_samples=val_samples,
    batch_size=int(best['batch_size']),
    tickers=TICKERS,
    orderflow_lookback=encoder_steps,
    num_workers=0,
    return_metadata=True,
)

base_model = CrackSpreadTFT(
    n_past_features=len(spread_feature_cols),
    n_known_features=len(known_temporal_cols),
    n_orderflow_feats=len(orderflow_cols),
    n_assets=len(TICKERS),
    encoder_steps=encoder_steps,
    decoder_steps=DECODER_STEPS,
    d_model=int(best['d_model']),
    n_heads=int(best['n_heads']),
    n_lstm_layers=int(best['n_lstm_layers']),
    dropout=float(best['dropout']),
    num_classes=4,
    vsn_temperature=float(best['vsn_temperature']),
    vsn_min_weight=float(best['vsn_min_weight']),
    vsn_entropy_weight=float(best['vsn_entropy_weight']),
    asset_temperature=float(best['asset_temperature']),
    asset_min_weight=float(best['asset_min_weight']),
    asset_entropy_weight=float(best['asset_entropy_weight']),
).to(device)
final_model = CrackSpreadPTP(
    base_model=base_model,
    ptp_temperature=float(best['ptp_temperature']),
    dropout=float(best['dropout']),
).to(device)

loss_fn = PTPLoss(
    ce_weight=float(best['ce_weight']),
    pnl_weight=float(best['pnl_weight']),
    profit_scale=float(best['profit_scale']),
    outer_threshold=float(best['outer_threshold']),
    tc_cost=float(best['tc_cost']),
    direction_weight=float(best['init_direction_weight']),
    reg_weight=float(best['init_reg_weight']),
    target_exposure=float(best['target_exposure']),
    use_sortino=True,
    downside_vol_weight=float(best['downside_vol_weight']),
    tc_in_sharpe=True,
    holding_weight=float(best['holding_weight']),
    exposure_asymmetry=float(best['exposure_asymmetry']),
).to(device)
optimizer = optim.AdamW(
    build_crack_ptp_param_groups(
        final_model,
        base_lr=float(best['learning_rate']),
        weight_decay=float(best['weight_decay']),
        ptp_lr_scale=float(best['ptp_lr_scale']),
    )
)
lr_scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=max(NUM_EPOCHS + 8, 20),
    eta_min=float(best['learning_rate']) * 0.05,
)
sharpe_sched = SharpeScheduler(
    warmup_epochs=WARMUP_EPOCHS,
    total_epochs=max(NUM_EPOCHS + 8, 20),
    initial_direction_weight=float(best['init_direction_weight']),
    final_direction_weight=float(best['final_direction_weight']),
    initial_reg_weight=float(best['init_reg_weight']),
    final_reg_weight=0.05,
    initial_target_exposure=0.2,
    final_target_exposure=float(best['target_exposure']),
    initial_holding_weight=0.0,
    final_holding_weight=float(best['holding_weight']),
)
scaler = torch.amp.GradScaler() if USE_AMP else None

result = fit_final_crack_ptp(
    model=final_model,
    train_loader=train_loader,
    val_loader=val_loader,
    loss_fn=loss_fn,
    optimizer=optimizer,
    scheduler=lr_scheduler,
    sharpe_scheduler=sharpe_sched,
    device=device,
    num_epochs=max(NUM_EPOCHS + 8, 20),
    warmup_epochs=WARMUP_EPOCHS,
    max_norm=float(best['max_norm']),
    aux_weight=float(best['aux_weight']),
    scaler=scaler,
)
if result.best_state is not None:
    final_model.load_state_dict(result.best_state)

print('Best final score:', result.best_score)
print('Best validation metrics:')
print(json.dumps(result.best_metrics, indent=2, default=float))

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes[0, 0].plot(result.history['train_loss'], label='train')
axes[0, 0].plot(result.history['val_loss'], label='val')
axes[0, 0].set_title('Loss')
axes[0, 0].legend()
axes[0, 1].plot(result.history['val_score'])
axes[0, 1].set_title('Validation selection score')
axes[1, 0].plot(result.history['val_sharpe'], label='sharpe')
axes[1, 0].plot(result.history['val_sortino'], label='sortino')
axes[1, 0].legend()
axes[1, 0].set_title('Trading metrics')
axes[1, 1].plot(result.history['val_cls_acc'])
axes[1, 1].set_title('Validation class accuracy')
plt.tight_layout()

save_dict = {
    'model_state_dict': final_model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'best_metrics': result.best_metrics,
    'best_score': result.best_score,
    'best_params': best,
    'target_col': TARGET_COL,
    'tickers': TICKERS,
    'bar_minutes': BAR_MINUTES,
    'forecast_horizon_minutes': FORECAST_HORIZON_MINUTES,
    'spread_feature_cols': spread_feature_cols,
    'known_temporal_cols': known_temporal_cols,
    'orderflow_cols': orderflow_cols,
}
torch.save(save_dict, RESULTS_PATH / f'{STUDY_NAME}_best_model.pth')


In [ ]:
from notebooks.crack_spread_tft_ptp_support import batch_to_device

final_model.eval()
all_positions = []
all_returns = []
all_logits = []
all_timestamps = []

with torch.no_grad():
    for batch in val_loader:
        inputs, targets = batch_to_device(batch, device)
        position, logits, _ = final_model(**inputs)
        all_positions.append(position.detach().cpu().numpy().reshape(-1))
        all_returns.append(targets.detach().cpu().numpy().reshape(-1))
        all_logits.append(logits.detach().cpu().numpy())
        all_timestamps.extend(batch['_anchor_ts'])

val_positions = np.concatenate(all_positions)
val_returns = np.concatenate(all_returns)
val_logits = np.concatenate(all_logits)
val_metrics = compute_trading_metrics(val_positions, val_returns, logits=val_logits, outer_threshold=float(best['outer_threshold']))
print(json.dumps(val_metrics, indent=2, default=float))

bt_df = pd.DataFrame({
    'anchor_ts': pd.to_datetime(all_timestamps),
    'position': val_positions,
    'forward_return': val_returns,
})
bt_df['strategy_return'] = bt_df['position'] * bt_df['forward_return']
bt_df['cum_strategy_return'] = bt_df['strategy_return'].cumsum()
bt_df['cum_forward_return'] = bt_df['forward_return'].cumsum()

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
axes[0].plot(bt_df['anchor_ts'], bt_df['cum_strategy_return'], label='strategy')
axes[0].plot(bt_df['anchor_ts'], bt_df['cum_forward_return'], label='buy_hold', alpha=0.7)
axes[0].legend()
axes[0].set_title('Validation cumulative returns')
axes[1].plot(bt_df['anchor_ts'], bt_df['position'], alpha=0.8)
axes[1].set_title('Validation position path')
plt.tight_layout()

bt_df.to_csv(RESULTS_PATH / f'{STUDY_NAME}_validation_backtest.csv', index=False)
